# **END-TO-END SUPPLY CHAIN & PROFITABILITY ANALYSIS**

This project analyzes the Global Superstore dataset to evaluate supply chain performance and profitability. The workflow mirrors a complete EDA portfolio project with cleaning, feature engineering, KPI development, shipping efficiency analysis, and profitability diagnostics.


**IMPORT NECESSARY LIBRARIES**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


**LOADING THE DATASET**


In [ ]:
candidate_paths = [
    Path("../data/Sample - Superstore.csv"),
    Path("data/Sample - Superstore.csv"),
    Path("Sample - Superstore.csv"),
    Path(r"C:/Users/Hp/Downloads/Sample - Superstore.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Sample - Superstore.csv not found. Place it in data/ or project root.")

df = pd.read_csv(csv_path, encoding_errors="ignore")
print(f"Loaded dataset from: {csv_path}")
print(f"Shape: {df.shape}")
df.head()


**INITIAL DATA UNDERSTANDING**


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


**DATA CLEANING AND PREPROCESSING**


In [ ]:
df.columns = [col.strip() for col in df.columns]

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

null_summary = df.isna().sum().sort_values(ascending=False)
null_summary[null_summary > 0]


In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Duplicate rows removed: {before - after}")


**FEATURE ENGINEERING**


In [ ]:
df["Delivery Time (Days)"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin %"] = np.where(df["Sales"] != 0, (df["Profit"] / df["Sales"]) * 100, 0)
df["Order YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)

df[["Order Date", "Ship Date", "Delivery Time (Days)", "Sales", "Profit", "Profit Margin %"]].head()


**KPI DEVELOPMENT**


In [ ]:
kpis = {
    "Total Sales": df["Sales"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Overall Profit Margin %": (df["Profit"].sum() / df["Sales"].sum()) * 100,
    "Average Delivery Time (Days)": df["Delivery Time (Days)"].mean(),
    "Total Orders": df["Order ID"].nunique(),
    "Total Customers": df["Customer ID"].nunique(),
}
pd.DataFrame({"KPI": list(kpis.keys()), "Value": list(kpis.values())})


**SHIPPING EFFICIENCY ACROSS SHIP MODES**


In [ ]:
shipping_eff = (
    df.groupby("Ship Mode", as_index=False)
      .agg(
          Avg_Delivery_Days=("Delivery Time (Days)", "mean"),
          Total_Orders=("Order ID", "nunique"),
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum")
      )
      .sort_values("Avg_Delivery_Days")
)
shipping_eff


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=shipping_eff, x="Ship Mode", y="Avg_Delivery_Days", palette="viridis")
plt.title("Average Delivery Time by Ship Mode")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


**PROFITABILITY ANALYSIS (REGIONS, CATEGORIES, PRODUCTS)**


In [ ]:
region_profit = (
    df.groupby("Region", as_index=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
      .sort_values("Total_Profit")
)
region_profit


In [ ]:
loss_products = (
    df.groupby(["Product ID", "Product Name"], as_index=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
      .query("Total_Profit < 0")
      .sort_values("Total_Profit")
)
loss_products.head(15)


In [ ]:
subcat_profit = (
    df.groupby("Sub-Category", as_index=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
      .sort_values("Total_Profit")
)
fig = px.bar(
    subcat_profit,
    x="Sub-Category",
    y="Total_Profit",
    title="Profitability by Sub-Category",
    color="Total_Profit",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(xaxis_tickangle=45)
fig.show()


**TREND ANALYSIS**


In [ ]:
monthly = (
    df.groupby("Order YearMonth", as_index=False)
      .agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
)
fig = px.line(
    monthly,
    x="Order YearMonth",
    y=["Total_Sales", "Total_Profit"],
    title="Monthly Sales and Profit Trend",
)
fig.show()


**POWER BI DASHBOARD COMPONENTS (RECOMMENDED)**


In [ ]:
dashboard_components = [
    "KPI Cards: Total Sales, Total Profit, Profit Margin %, Avg Delivery Time",
    "Shipping Efficiency: Avg Delivery Time by Ship Mode",
    "Regional Performance: Sales vs Profit by Region",
    "Profitability Drilldown: Category > Sub-Category > Product",
    "Time Series: Monthly Sales and Profit",
]
pd.DataFrame({"Dashboard Components": dashboard_components})


**BUSINESS INSIGHTS**


In [ ]:
insights = [
    "Target high-margin regions and segments for growth-focused campaigns.",
    "Investigate loss-making products for pricing, bundling, or discontinuation decisions.",
    "Reduce slow ship-mode dependencies where delivery delays impact customer experience.",
    "Use monthly trends to optimize inventory and procurement planning cycles.",
]

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")
